# NB44 — KANSER G_LOW Missing-Flag Ablasyonu

NB44 -- KANSER G_LOW Flag Ablasyonu

Amac: NB43'un bulgusu -- KANSER panelinde G_LOW grubu (miss_ratio <= %50)
mean|phi|=0.260 ile G_HIGH'a (0.410) yakin guclu bir sinyal tasiyor. Mevcut
M3 stratejisi (>%50 icin flag) bu grubu flag'lemiyor -- potansiyel kayip sinyal.

Denenen: M3 (mevcut, sadece >%50 NaN icin is_missing flag) vs M3+phi-secici
(G_LOW icinde |phi|>0.15 olan sutunlara da ek olarak is_missing flag ekle).
Referans liste: results/v26_missing_flag_correlation/KANSER_LOW_flag_stats.csv

Champion konfigurasyon (NB32 P9_REVERSE_6040/catboost/with_fe, Boot-F1=0.730)
SABIT tutuluyor; degisen TEK eksen missing-flag stratejisi. Nested-parallelism
deadlock riskini onlemek icin (bkz. NB46 metodolojik notu) LightGBM/CatBoost
tek-thread calistiriliyor, dis paralellik yok (bu script zaten sirali).

Degerlendirme protokolu (CLAUDE.md zorunlu unsurlari):
  - floor-F1 referansi (kendi test havuzunun prevalansindan)
  - f1_raw (train dengesi) vs f1_8020/mcc_8020 (final-realistic) ayrimi
  - %50/50 + %80/20 bootstrap (N=50, %95 CI) yan yana
  - train metrikleri (overfit kontrolu)

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, precision_score, recall_score, matthews_corrcoef,
    average_precision_score, confusion_matrix
)

from lightgbm import LGBMClassifier
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print('[UYARI] catboost bulunamadi, sadece lgbm kullanilacak')

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR

# --- Sabitler ---
PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
N_OOF_FOLDS = 5
BOOT_SEED = 123
PANEL_SPLIT_FRAC = 0.50
HIGH_MISS_THR = 0.50
PHI_THR = 0.15

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v28_kanser_missing_flag_ablation')
os.makedirs(RESULTS_DIR, exist_ok=True)
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f'SEED={SEED}, PROJECT_ROOT={PROJECT_ROOT}')
print(f'Results -> {RESULTS_DIR}')

ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

In [ ]:
# ============================================================================
# Cell: Veri Yukleme + Sutun Temizligi (NB32 ile ayni pattern)
# ============================================================================
data_dir = os.path.join(PROJECT_ROOT, 'data', 'real_data')
df_master = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_MASTER.csv'))
df_kanser = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_KANSER.csv'))
df_cftr = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_CFTR.csv'))
df_pah = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_PAH.csv'))

print(f'MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})')
print(f'KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})')

df_combined_raw = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)

feat_cols_raw = [c for c in df_kanser.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_kanser, df_master, feat_cols_raw, TARGET)
if dup_ids:
    df_kanser = df_kanser[~df_kanser[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f'KANSER: {len(dup_ids)} birebir-ayni satir drop -> {df_kanser.shape}')
else:
    print('KANSER: birebir-ayni satir yok')

constant_cols = [c for c in feat_cols_raw if df_master[c].nunique(dropna=False) <= 1]

def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, feat_cols_raw)
drop_cols = set(constant_cols) | dup_drop
keep_cols = [c for c in feat_cols_raw if c not in drop_cols]
print(f'Constant: {len(constant_cols)}, Dup pairs: {len(dup_pairs)} -> drop {len(dup_drop)}')
print(f'Toplam drop: {len(drop_cols)}, Kalan feature: {len(keep_cols)}')

for _df in [df_master, df_kanser, df_cftr, df_pah]:
    for c in drop_cols:
        if c in _df.columns:
            _df.drop(columns=[c], inplace=True)

df_combined = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)
print(f'\nFinal: MASTER={df_master.shape}, COMBINED={df_combined.shape}, KANSER={df_kanser.shape}')

def panel_5050_split(df):
    pos = df[df[TARGET]==1].sample(frac=1.0, random_state=SEED)
    neg = df[df[TARGET]==0].sample(frac=1.0, random_state=SEED)
    npos = int(round(len(pos) * PANEL_SPLIT_FRAC))
    nneg = int(round(len(neg) * PANEL_SPLIT_FRAC))
    tr = pd.concat([pos.iloc[:npos], neg.iloc[:nneg]]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    te = pd.concat([pos.iloc[npos:], neg.iloc[nneg:]]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return tr, te

kanser_train, kanser_test = panel_5050_split(df_kanser)
y_kanser_train = kanser_train[TARGET].values
y_kanser_test = kanser_test[TARGET].values
print(f'\nKANSER train: {kanser_train.shape} (pos={y_kanser_train.sum()}, neg={(y_kanser_train==0).sum()})')
print(f'KANSER test:  {kanser_test.shape} (pos={y_kanser_test.sum()}, neg={(y_kanser_test==0).sum()})')
assert y_kanser_test.sum() > 0 and (y_kanser_test==0).sum() > 0, 'Split hatasi!'

In [ ]:
# ============================================================================
# Cell: G_LOW phi-secici sutun listesi (NB43 referansindan)
# ============================================================================
low_stats_path = os.path.join(PROJECT_ROOT, 'results', 'v26_missing_flag_correlation',
                                'KANSER_LOW_flag_stats.csv')
low_stats = pd.read_csv(low_stats_path)
low_stats = low_stats[low_stats['col'].isin(keep_cols)]
phi_selective_cols = low_stats.loc[low_stats['abs_phi'] > PHI_THR, 'col'].tolist()
print(f'\nG_LOW referans satir: {len(low_stats)}, |phi|>{PHI_THR} secici flag adayi: {len(phi_selective_cols)}')
print(f'Ornekler: {phi_selective_cols[:10]}')

In [ ]:
# ============================================================================
# Cell: Missing-Flag Stratejileri (M3 mevcut vs M3+phi-secici)
# ============================================================================
def fit_preprocessor(train_df, keep_cols, target=TARGET, extra_flag_cols=None):
    """M3 (>%50 NaN flag) + opsiyonel G_LOW phi-secici ek flag kumesi."""
    X = train_df[keep_cols].copy()
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    extra_flags = [c for c in (extra_flag_cols or []) if c in num_cols and c not in high_miss]
    medians = X[num_cols].median()
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "extra_flags": extra_flags,
        "medians": medians, "le_maps": le_maps, "keep_cols": keep_cols
    }

def transform_X(df, prep):
    kc = prep["keep_cols"]
    X = df[kc].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    for c in prep["high_miss"] + prep["extra_flags"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return X

print("\nMissing-flag stratejileri hazir: M3 (baseline), M3_PHI (G_LOW phi-secici ek)")

In [ ]:
# ============================================================================
# Cell: Feature Engineering (NB16/NB32 Cell 4/7 -- Grantham/BLOSUM62/stopgain)
# ============================================================================
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}

def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a))

_B62_RAW = '''A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4'''
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for ri, line in enumerate(_B62_RAW.strip().split("\n")):
    toks = line.split()
    row_aa = toks[0][0]
    vals = [toks[0][1:]] + toks[1:]
    for ci, tok in enumerate(vals):
        col_aa = _ORDER[ri + ci]
        v = int(tok[1:] if tok[0].isalpha() else tok)
        _B62[(row_aa, col_aa)] = v; _B62[(col_aa, row_aa)] = v

def blosum62(a, b):
    return _B62.get((a, b), 0)

def add_fe(df):
    out = df.copy()
    a1 = out["AA_1"].astype("object"); a2 = out["AA_2"].astype("object")
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    def _nonstd(v):
        return 0 if (isinstance(v, str) and v in STANDARD_AA) else 1
    out["fe_aa_nonstandard"] = (a1.map(_nonstd) | a2.map(_nonstd)).astype(int)
    def _gr(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return grantham(x, y)
        return -1
    def _bl(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return blosum62(x, y)
        return 0
    out["fe_grantham"] = out.apply(_gr, axis=1).astype(float)
    out["fe_blosum62"] = out.apply(_bl, axis=1).astype(float)
    return out

FE_NEW_COLS = ["fe_aa_stopgain", "fe_aa_nonstandard", "fe_grantham", "fe_blosum62"]
print(f"FE hazir. Yeni sutunlar: {FE_NEW_COLS}")

In [ ]:
# ============================================================================
# Cell: Champion Pool (P9_REVERSE_6040 -- COMBINED %60B/%40P) -- NB32 ile ayni
# ============================================================================
def build_champion_pool():
    comb_neg = df_combined[df_combined[TARGET]==0]
    comb_pos = df_combined[df_combined[TARGET]==1]
    n_neg = len(comb_neg)
    n_pos = max(1, int(round(n_neg * 0.40 / 0.60)))
    pos_sample = comb_pos.sample(n=min(n_pos, len(comb_pos)), random_state=SEED)
    pool = pd.concat([comb_neg, pos_sample]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return pool

champion_pool = build_champion_pool()
pos_p = champion_pool[TARGET].sum(); neg_p = (champion_pool[TARGET]==0).sum()
print(f'\nChampion pool P9_REVERSE_6040: {champion_pool.shape} (pos={pos_p}, neg={neg_p}, '
      f'benign_frac={neg_p/(pos_p+neg_p):.3f})')

In [ ]:
# ============================================================================
# Cell: Tree Model Helper (catboost, tek-thread -- NB46 deadlock dersine gore)
# ============================================================================
LGBM_PARAMS = {
    "n_estimators": 300, "num_leaves": 31, "learning_rate": 0.05,
    "min_child_samples": 20, "subsample": 0.8, "colsample_bytree": 0.8,
    "class_weight": "balanced",
    "random_state": SEED, "verbose": -1, "n_jobs": 1, "importance_type": "gain"
}
CB_PARAMS = {
    "iterations": 300, "depth": 6, "learning_rate": 0.05,
    "auto_class_weights": "Balanced",
    "random_seed": SEED, "verbose": 0, "thread_count": 1
}

def _prep_tree(X_df):
    cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
    Xn = X_df.copy()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, cat_cols, le_maps

def _apply_le(X_df, cat_cols, le_maps):
    Xn = X_df.copy()
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = le_maps[c]
        Xn[c] = Xn[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return Xn

def oof_catboost(X_train_df, y_train, X_test_df, n_splits=N_OOF_FOLDS):
    X_tr, cat_cols, le_maps = _prep_tree(X_train_df)
    X_te = _apply_le(X_test_df, cat_cols, le_maps)
    cat_idx = [list(X_tr.columns).index(c) for c in cat_cols]

    oof = np.zeros(len(y_train))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for tri, vai in skf.split(X_tr, y_train):
        m = CatBoostClassifier(**CB_PARAMS)
        m.fit(X_tr.iloc[tri], y_train[tri], cat_features=cat_idx, silent=True)
        oof[vai] = m.predict_proba(X_tr.iloc[vai])[:, 1]

    mf = CatBoostClassifier(**CB_PARAMS)
    mf.fit(X_tr, y_train, cat_features=cat_idx, silent=True)
    test_proba = mf.predict_proba(X_te)[:, 1]
    return oof, test_proba

In [ ]:
# ============================================================================
# Cell: Degerlendirme Altyapisi (NB32/NB39/NB46 ile birebir ayni protokol)
# ============================================================================
def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    return float(max(thr_scores, key=thr_scores.get))

def select_threshold_raw(y, prob):
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        thr_scores[thr] = _f1_pos(y, (prob >= thr).astype(int))
    return float(max(thr_scores, key=thr_scores.get))

def floor_f1(y):
    prev = float(np.mean(y))
    return 2 * prev / (1 + prev)

def eval_full(label, y_test, p_test, y_train, p_train):
    """f1_raw, f1_8020, mcc_8020 uclusu + 50/50 ve 80/20 bootstrap + floor + train metrikleri."""
    thr_raw = select_threshold_raw(y_train, p_train)
    thr_8020 = select_threshold_8020_robust(y_train, p_train)

    yp_raw = (p_test >= thr_raw).astype(int)
    f1_raw = _f1_pos(y_test, yp_raw)

    yp_8020_thr = (p_test >= thr_8020).astype(int)
    f1_5050 = _f1_pos(y_test, yp_8020_thr)  # mevcut panel dagilimi (~%50/50) uzerinde, 8020-esikli
    mcc_5050 = matthews_corrcoef(y_test, yp_8020_thr)
    prec_5050 = precision_score(y_test, yp_8020_thr, pos_label=1, zero_division=0)
    rec_5050 = recall_score(y_test, yp_8020_thr, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, yp_8020_thr, labels=[0,1]).ravel()
    auprc = average_precision_score(y_test, p_test) if len(np.unique(y_test)) > 1 else 0.0

    boot = bootstrap_8020(y_test, p_test, thr_8020)
    f1_8020 = boot["mean"]

    rng = np.random.RandomState(BOOT_SEED)
    mccs = []
    for _ in range(N_BOOT):
        yb, pb = _resample_8020(y_test, p_test, rng)
        mccs.append(matthews_corrcoef(yb, (pb >= thr_8020).astype(int)))
    mcc_8020 = float(np.mean(mccs))

    yp_train_8020 = (p_train >= thr_8020).astype(int)
    train_f1 = _f1_pos(y_train, yp_train_8020)
    train_mcc = matthews_corrcoef(y_train, yp_train_8020)

    floor = floor_f1(y_test)

    return {
        "label": label,
        "thr_raw": thr_raw, "thr_8020": thr_8020,
        "f1_raw": f1_raw,
        "f1_5050_8020thr": f1_5050, "mcc_5050_8020thr": mcc_5050,
        "prec_5050": prec_5050, "rec_5050": rec_5050,
        "f1_8020_boot": f1_8020, "f1_8020_std": boot["std"],
        "f1_8020_ci_lo": boot["lo"], "f1_8020_ci_hi": boot["hi"],
        "mcc_8020_boot": mcc_8020,
        "auprc": auprc, "fp": int(fp), "fn": int(fn), "tp": int(tp), "tn": int(tn),
        "floor_f1": floor, "gecti_mi_floor": bool(f1_8020 > floor),
        "train_f1": train_f1, "train_mcc": train_mcc,
        "train_test_gap": train_f1 - f1_8020,
    }

print("\nDegerlendirme altyapisi hazir (f1_raw / f1_8020 / mcc_8020 + floor + train-gap).")

In [ ]:
# ============================================================================
# Cell: Ablasyon Calistir -- M3 vs M3_PHI (with_fe sabit, champion recete)
# ============================================================================
STRATEGIES = {
    "M3_baseline": [],
    "M3_PHI_selective": phi_selective_cols,
}

all_results = {}
for strat_name, extra_flags in STRATEGIES.items():
    print(f"\n{'='*70}\n[{strat_name}] extra_flags={len(extra_flags)}\n{'='*70}")

    pool_fe = add_fe(champion_pool)
    test_fe = add_fe(kanser_test)
    train_fe = add_fe(kanser_train)
    fe_keep = keep_cols + FE_NEW_COLS

    prep = fit_preprocessor(pool_fe, keep_cols=fe_keep, extra_flag_cols=extra_flags)
    X_pool = transform_X(pool_fe, prep)
    X_test = transform_X(test_fe, prep)
    X_train_panel = transform_X(train_fe, prep)
    y_pool = pool_fe[TARGET].values

    if HAS_CATBOOST:
        oof, test_proba = oof_catboost(X_pool, y_pool, X_test)
        _, train_panel_proba = oof_catboost(X_pool, y_pool, X_train_panel)
    else:
        # yedek: lgbm (catboost yoksa)
        skf = StratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=SEED)
        X_tr_p, cat_cols, le_maps = _prep_tree(X_pool)
        X_te_p = _apply_le(X_test, cat_cols, le_maps)
        for c in cat_cols:
            X_tr_p[c] = X_tr_p[c].astype("category")
            X_te_p[c] = pd.Categorical(X_te_p[c], categories=X_tr_p[c].cat.categories)
        oof = np.zeros(len(y_pool))
        for tri, vai in skf.split(X_tr_p, y_pool):
            m = LGBMClassifier(**LGBM_PARAMS)
            m.fit(X_tr_p.iloc[tri], y_pool[tri], categorical_feature=cat_cols)
            oof[vai] = m.predict_proba(X_tr_p.iloc[vai])[:, 1]
        mf = LGBMClassifier(**LGBM_PARAMS)
        mf.fit(X_tr_p, y_pool, categorical_feature=cat_cols)
        test_proba = mf.predict_proba(X_te_p)[:, 1]
        train_panel_proba = test_proba  # placeholder, catboost yoksa atlanir

    res = eval_full(strat_name, y_kanser_test, test_proba, y_pool, oof)
    res["n_extra_flags"] = len(extra_flags)
    res["n_pool"] = len(pool_fe)
    all_results[strat_name] = res

    print(f"  f1_raw={res['f1_raw']:.4f}  f1_8020_boot={res['f1_8020_boot']:.4f} "
          f"+/-{res['f1_8020_std']:.3f} [CI {res['f1_8020_ci_lo']:.3f}-{res['f1_8020_ci_hi']:.3f}]  "
          f"mcc_8020={res['mcc_8020_boot']:.4f}  floor={res['floor_f1']:.4f}  "
          f"gecti_mi={res['gecti_mi_floor']}  train_test_gap={res['train_test_gap']:.4f}")

In [ ]:
# ============================================================================
# Cell: Sonuc Derleme + CSV + Gorsellestirme
# ============================================================================
res_df = pd.DataFrame(all_results.values())
res_df.to_csv(os.path.join(RESULTS_DIR, 'nb44_kanser_missing_flag_ablation_results.csv'), index=False)
print(f"\nSonuclar kaydedildi: {RESULTS_DIR}/nb44_kanser_missing_flag_ablation_results.csv")
print(res_df[["label", "n_extra_flags", "f1_raw", "f1_8020_boot", "f1_8020_ci_lo",
              "f1_8020_ci_hi", "mcc_8020_boot", "floor_f1", "gecti_mi_floor",
              "train_test_gap"]].to_string(index=False))

baseline_boot = all_results["M3_baseline"]["f1_8020_boot"]
phi_boot = all_results["M3_PHI_selective"]["f1_8020_boot"]
delta = phi_boot - baseline_boot
print(f"\nDelta (M3_PHI - M3_baseline) Boot-F1: {delta:+.4f}")

reference_champion_boot = 0.730  # NB32 P9_REVERSE_6040_catboost_with_fe
print(f"Referans sampiyon (NB32 P9_catboost_with_fe): {reference_champion_boot:.4f}")
print(f"Bu deney en iyisi: {max(baseline_boot, phi_boot):.4f} "
      f"({'M3_PHI_selective' if phi_boot > baseline_boot else 'M3_baseline'})")

fig, ax = plt.subplots(figsize=(7, 5))
labels = list(all_results.keys())
means = [all_results[l]["f1_8020_boot"] for l in labels]
los = [all_results[l]["f1_8020_ci_lo"] for l in labels]
his = [all_results[l]["f1_8020_ci_hi"] for l in labels]
errs = [[m - lo for m, lo in zip(means, los)], [hi - m for m, hi in zip(means, his)]]
ax.bar(labels, means, yerr=errs, capsize=6, color=['#4C72B0', '#DD8452'])
ax.axhline(all_results["M3_baseline"]["floor_f1"], color='red', linestyle='--', label='Floor-F1')
ax.axhline(reference_champion_boot, color='green', linestyle=':', label='NB32 Referans (0.730)')
ax.set_ylabel('Boot-F1 (%80/20, %95 CI)')
ax.set_title('NB44 -- KANSER Missing-Flag Ablasyonu (M3 vs M3+phi-secici)')
ax.legend()
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'nb44_ablation_comparison.png')
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"Gorsel kaydedildi: {fig_path}")

In [ ]:
# ============================================================================
# Cell: Ozet JSON
# ============================================================================
summary = {
    "notebook": "NB44",
    "date": "2026-07-28",
    "hypothesis": "KANSER G_LOW phi-secici missing-flag ablasyonu",
    "phi_threshold": PHI_THR,
    "n_phi_selective_cols": len(phi_selective_cols),
    "champion_pool": "P9_REVERSE_6040 (COMBINED %60B/%40P) + catboost + with_fe (NB32 recetesi sabit)",
    "reference_champion": {
        "name": "P9_REVERSE_6040_catboost_with_fe_NB32",
        "boot_f1": reference_champion_boot
    },
    "results": {k: {kk: vv for kk, vv in v.items() if kk != "label"} for k, v in all_results.items()},
    "delta_phi_vs_baseline": delta,
    "verdict": (
        "PHI-secici flag ekleme kayda deger kazanc sagladi" if delta > 0.01 else
        "PHI-secici flag ekleme zararli (M3 baseline'in altinda)" if delta < -0.01 else
        "PHI-secici flag ekleme anlamli fark yaratmadi (gurultu bandinda)"
    )
}
with open(os.path.join(RESULTS_DIR, 'nb44_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"\nOzet JSON kaydedildi: {RESULTS_DIR}/nb44_summary.json")
print(f"\nSonuc: {summary['verdict']}")